# Multi-Asset Performance and Risk Terminal

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdiArora707/multi-asset-risk-terminal/blob/main/notebooks/Multi_Asset_Performance_Risk_Terminal.ipynb)

**Equities · Bonds · Commodities · Real Estate · Cash**

This Colab is the research interface to a reusable, tested Python analytics package. It downloads
adjusted ETF prices and FRED macro series, constructs a financing-aware multi-asset portfolio,
separates beta from alpha, measures tail risk and return concentration, and exports an interactive
HTML tear sheet.

> Educational research only—not investment advice. Review the provider terms before using market
> data outside a personal/research setting.

## 0. Colab setup

When running in Google Colab, the setup cell clones the published GitHub repository. A local Jupyter
session automatically uses the checked-out project. Dependency installation is explicit and repeatable.

In [5]:
#@title Install the project
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AdiArora707/multi-asset-risk-terminal.git"  #@param {type:"string"}

if IN_COLAB:
    project_dir = Path("/content/multi-asset-risk-terminal")
    if not project_dir.exists():
        if "YOUR_USERNAME" in REPO_URL or not REPO_URL.startswith("https://github.com/"):
            raise ValueError("REPO_URL must point to a valid GitHub repository.")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project_dir)], check=True)
    else:
        # Colab runtimes can survive notebook reloads. Fast-forward the existing
        # clone so an old package cannot silently generate a stale report.
        subprocess.run(
            ["git", "-C", str(project_dir), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
    os.chdir(project_dir)
else:
    project_dir = Path.cwd()
    if not (project_dir / "pyproject.toml").exists():
        raise RuntimeError("Start Jupyter from the repository root.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    check=True,
)
try:
    commit = subprocess.check_output(
        ["git", "-C", str(project_dir), "rev-parse", "--short", "HEAD"],
        text=True,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    commit = "local checkout"
print(f"Project ready: {project_dir} (commit {commit})")

Project ready: /content/multi-asset-risk-terminal (commit 14aa742)


## 1. Imports and display configuration

The notebook imports the public project API. All finance formulas live in `src/`, where they can be
unit-tested and reused by the command-line job.

In [6]:
import os
import sys
from IPython.display import HTML, display
import pandas as pd
import plotly.io as pio

# Ensure the project's source directory is on the Python path
# project_dir is defined in cell terminal-cell-02
if 'project_dir' in locals() or 'project_dir' in globals():
    sys.path.append(str(project_dir / "src"))
else:
    # Fallback if project_dir is not in scope (e.g., cell executed out of order)
    sys.path.append('/content/multi-asset-risk-terminal/src')

from multi_asset_terminal import TerminalConfig, export_analysis_artifacts, run_analysis
from multi_asset_terminal.visualizations import monthly_return_matrix

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
pio.templates.default = "plotly_white"

## 2. Investment policy and run configuration

Weights may sum above 100%. Any negative residual is treated as borrowing and charged the daily
FRED risk-free rate. The default portfolio is unlevered and rebalanced monthly.

In [7]:
#@title Configure the analysis
START_DATE = "2015-01-01"  #@param {type:"date"}
END_DATE = ""  #@param {type:"string"}
BENCHMARK = "SPY"  #@param {type:"string"}
ROLLING_WINDOW = 252  #@param {type:"integer"}
REFRESH_DATA = False  #@param {type:"boolean"}

assets = {
    "US Equity": "VTI",
    "International Equity": "VXUS",
    "US Bonds": "BND",
    "Gold": "GLD",
    "Real Estate": "VNQ",
    "Cash ETF": "BIL",
}
weights = {
    "US Equity": 0.30,
    "International Equity": 0.15,
    "US Bonds": 0.25,
    "Gold": 0.10,
    "Real Estate": 0.10,
    "Cash ETF": 0.10,
}

config = TerminalConfig(
    assets=assets,
    weights=weights,
    benchmark=BENCHMARK,
    start=START_DATE,
    end=END_DATE or None,
    rebalance_frequency="M",
    rolling_window=ROLLING_WINDOW,
)
print(f"Gross exposure: {config.gross_exposure:.1%}")
print(f"Net risky exposure: {config.net_exposure:.1%}")
display(pd.DataFrame({"Ticker": config.assets, "Target Weight": config.weights}))

Gross exposure: 100.0%
Net risky exposure: 100.0%


,Ticker,Target Weight
US Equity,VTI,0.3000
International Equity,VXUS,0.1500
US Bonds,BND,0.2500
Gold,GLD,0.1000
Real Estate,VNQ,0.1000
Cash ETF,BIL,0.1000


## 3. Data collection pipeline

The run uses adjusted Yahoo Finance closes (`auto_adjust=True`) plus FRED `DGS3MO` and `CPIAUCSL`.
The cell securely reads `FRED_API_KEY` from Colab Secrets or the local environment. Without a key,
the official FRED CSV route is attempted before the disclosed constant-rate fallback. Downloads are
retried, validated, and cached by parameters.

In [8]:
# Secure FRED authentication: never paste a key into a public notebook.
FRED_API_KEY = os.getenv("FRED_API_KEY")
if IN_COLAB and not FRED_API_KEY:
    try:
        from google.colab import userdata
        FRED_API_KEY = userdata.get("FRED_API_KEY")
    except Exception:
        print("FRED_API_KEY was not found in Colab Secrets; the public CSV route will be tried.")

print("FRED authenticated API:", "enabled" if FRED_API_KEY else "not configured")

results = run_analysis(
    config,
    refresh=REFRESH_DATA,
    fred_api_key=FRED_API_KEY,
)

print(f"Price observations: {len(results.prices):,}")
print(f"Analysis observations: {len(results.evaluation_returns):,}")
print(f"Sample: {results.evaluation_returns.index.min().date()} to "
      f"{results.evaluation_returns.index.max().date()}")
if results.warnings:
    for warning in results.warnings:
        print(f"WARNING: {warning}")
display(results.prices.tail())
display(results.macro.tail())

FRED authenticated API: enabled
Price observations: 2,924
Analysis observations: 2,923
Sample: 2015-01-05 to 2026-08-19


,US Equity,International Equity,US Bonds,Gold,Real Estate,Cash ETF,Benchmark
Date,,,,,,,
2026-08-13,384.3000,87.7200,72.4800,398.9600,98.5800,91.5100,777.8800
2026-08-14,383.8500,87.7000,72.3100,401.4800,98.8300,91.5300,776.3400
2026-08-17,382.1300,87.8700,72.1600,405.4900,97.9800,91.5500,772.6700
2026-08-18,379.0400,86.4400,72.2200,398.5500,97.6200,91.5600,767.4500
2026-08-19,379.9900,86.9700,72.5600,413.8400,98.6100,91.5700,769.0600


,annual_risk_free_rate,daily_risk_free_rate,inflation_yoy
Date,,,
2026-08-13,0.0387,0.0002,0.0373
2026-08-14,0.0386,0.0002,0.0373
2026-08-17,0.0387,0.0002,0.0354
2026-08-18,0.0386,0.0002,0.0354
2026-08-19,0.0386,0.0002,0.0354


## 4. Data cleaning and feature engineering

The common calendar uses complete observations and never forward-fills prices. This avoids creating
false zero returns. Simple returns drive compounding; log returns are retained for additive and
distributional research. Annual Treasury yields are converted into equivalent daily compounded rates,
and CPI levels become trailing 12-month inflation.

In [9]:
quality = pd.DataFrame({
    "Price rows": results.prices.count(),
    "Missing prices": results.prices.isna().sum(),
    "Simple return rows": results.simple_returns.count(),
    "Mean daily return": results.simple_returns.mean(),
    "Daily volatility": results.simple_returns.std(),
})
display(quality)
display(results.simple_returns.tail())
display(results.log_returns.tail())

,Price rows,Missing prices,Simple return rows,Mean daily return,Daily volatility
US Equity,2924,0,2923,0.0006,0.0113
International Equity,2924,0,2923,0.0004,0.0109
US Bonds,2924,0,2923,0.0001,0.0033
Gold,2924,0,2923,0.0005,0.0102
Real Estate,2924,0,2923,0.0003,0.0127
Cash ETF,2924,0,2923,0.0001,0.0002
Benchmark,2924,0,2923,0.0006,0.0111


,US Equity,International Equity,US Bonds,Gold,Real Estate,Cash ETF,Benchmark
Date,,,,,,,
2026-08-13,0.0065,0.0021,0.0028,-0.0147,0.0131,0.0000,0.0070
2026-08-14,-0.0012,-0.0002,-0.0023,0.0063,0.0025,0.0002,-0.0020
2026-08-17,-0.0045,0.0019,-0.0021,0.0100,-0.0086,0.0002,-0.0047
2026-08-18,-0.0081,-0.0163,0.0008,-0.0171,-0.0037,0.0001,-0.0068
2026-08-19,0.0025,0.0061,0.0047,0.0384,0.0101,0.0001,0.0021


,US Equity,International Equity,US Bonds,Gold,Real Estate,Cash ETF,Benchmark
Date,,,,,,,
2026-08-13,0.0064,0.0021,0.0028,-0.0148,0.0130,0.0000,0.0070
2026-08-14,-0.0012,-0.0002,-0.0023,0.0063,0.0025,0.0002,-0.0020
2026-08-17,-0.0045,0.0019,-0.0021,0.0099,-0.0086,0.0002,-0.0047
2026-08-18,-0.0081,-0.0164,0.0008,-0.0173,-0.0037,0.0001,-0.0068
2026-08-19,0.0025,0.0061,0.0047,0.0376,0.0101,0.0001,0.0021


## 5. Portfolio construction and reconciliation

Weights reset at the first observation of each month and drift between rebalances. `Financing` is
positive cash or negative borrowing. Daily asset contributions must sum exactly to each portfolio
return—this is an important production invariant.

In [10]:
path = results.portfolio_path
reconciliation_error = (
    path.contributions.sum(axis=1) - path.returns
).abs().max()

print(f"Maximum contribution reconciliation error: {reconciliation_error:.3e}")
display(path.weights.tail())
display(path.cash_weight.tail().to_frame())
display(path.contributions.tail())

Maximum contribution reconciliation error: 0.000e+00


,US Equity,International Equity,US Bonds,Gold,Real Estate,Cash ETF
Date,,,,,,
2026-08-13,0.3036,0.1515,0.2450,0.1063,0.0960,0.0977
2026-08-14,0.3047,0.1514,0.2450,0.1045,0.0970,0.0974
2026-08-17,0.3044,0.1514,0.2444,0.1052,0.0972,0.0975
2026-08-18,0.3034,0.1519,0.2443,0.1064,0.0965,0.0976
2026-08-19,0.3031,0.1504,0.2462,0.1053,0.0968,0.0983


,Financing
Date,
2026-08-13,0.0000
2026-08-14,0.0000
2026-08-17,0.0000
2026-08-18,0.0000
2026-08-19,0.0000


,US Equity,International Equity,US Bonds,Gold,Real Estate,Cash ETF,Financing
Date,,,,,,,
2026-08-13,0.0020,0.0003,0.0007,-0.0016,0.0013,0.0000,0.0000
2026-08-14,-0.0004,-0.0000,-0.0006,0.0007,0.0002,0.0000,0.0000
2026-08-17,-0.0014,0.0003,-0.0005,0.0011,-0.0008,0.0000,0.0000
2026-08-18,-0.0025,-0.0025,0.0002,-0.0018,-0.0004,0.0000,0.0000
2026-08-19,0.0008,0.0009,0.0012,0.0040,0.0010,0.0000,0.0000


## 6. Standardized performance and risk statistics

All return streams use the same CAGR, volatility, downside, drawdown, benchmark-relative, higher-
moment, and empirical tail-loss functions. Percentage metrics are formatted only for presentation;
the underlying tables remain numeric and export cleanly.

In [11]:
percent_metrics = [
    "Cumulative Return", "CAGR", "Annualized Volatility",
    "Annualized Downside Deviation", "Max Drawdown", "Jensen Alpha",
    "Tracking Error", "Hit Rate", "Best Day", "Worst Day",
    "CAGR Without Best 10 Days", "Best 10 Days / Positive Returns",
]
display(
    results.metrics.style
    .format({column: "{:.2%}" for column in percent_metrics})
    .format({
        "Sharpe Ratio": "{:.2f}", "Sortino Ratio": "{:.2f}",
        "Calmar Ratio": "{:.2f}", "Beta": "{:.2f}", "R-squared": "{:.2f}",
    })
    .background_gradient(subset=["Sharpe Ratio", "Sortino Ratio"], cmap="RdYlGn")
)

,Observations,Cumulative Return,CAGR,Annualized Volatility,Annualized Downside Deviation,Sharpe Ratio,Sortino Ratio,Calmar Ratio,Max Drawdown,Max Drawdown Duration,Skewness,Excess Kurtosis,Historical VaR 95%,Historical CVaR 95%,Tail Ratio,Profit Factor,Hit Rate,Best Day,Worst Day,CAGR Without Best 10 Days,Best 10 Days / Positive Returns,Beta,Jensen Alpha,R-squared,Tracking Error,Information Ratio,Up Capture,Down Capture
Portfolio,2923.000000,1.445646,0.080006,0.098556,0.071237,0.62,0.86,0.38,-0.211963,543.000000,-0.795681,17.963072,0.008994,0.014500,1.018198,1.170433,0.556620,0.050169,-0.071579,0.049507,0.051941,0.52,-0.003618,0.85,0.092848,-0.687666,0.520282,0.512849
Benchmark,2923.000000,3.532205,0.138895,0.175937,0.125286,0.71,1.00,0.41,-0.337173,488.000000,-0.307420,14.057339,0.016531,0.026689,0.943640,1.172819,0.547383,0.105019,-0.109423,0.077991,0.057597,1.00,0.000000,1.00,0.000000,nan,1.000000,1.000000
US Equity,2923.000000,3.339493,0.134644,0.179128,0.127946,0.68,0.95,0.38,-0.350003,492.000000,-0.345548,14.135083,0.016763,0.027093,0.954430,1.164516,0.545330,0.101456,-0.113809,0.073755,0.056680,1.01,-0.004924,0.99,0.016116,-0.196938,1.014481,1.020924
International Equity,2923.000000,1.526991,0.083052,0.172727,0.125896,0.43,0.59,0.23,-0.359723,706.000000,-0.844380,12.755406,0.015995,0.025318,1.003683,1.105415,0.532330,0.083810,-0.111318,0.035761,0.046189,0.83,-0.029660,0.72,0.095828,-0.531018,0.840435,0.873464
US Bonds,2923.000000,0.229849,0.017965,0.052964,0.038736,-0.03,-0.05,0.10,-0.185821,1515.000000,-0.913071,36.516134,0.004744,0.007392,0.971389,1.070688,0.517961,0.042201,-0.054385,0.002538,0.053066,0.04,-0.006145,0.01,0.177666,-0.712517,0.032084,0.014141
Gold,2923.000000,2.627630,0.117282,0.161741,0.113964,0.64,0.91,0.44,-0.264045,897.000000,-0.446844,6.671594,0.015544,0.023446,1.036514,1.145941,0.531646,0.063587,-0.102742,0.076699,0.038866,0.06,0.100037,0.00,0.230634,-0.093659,0.122244,-0.006422
Real Estate,2923.000000,0.893043,0.056460,0.202370,0.148505,0.27,0.37,0.13,-0.423982,1114.000000,-1.051326,19.870917,0.018763,0.029637,0.953898,1.072167,0.531646,0.089967,-0.177277,-0.002920,0.053335,0.82,-0.046162,0.50,0.146081,-0.480006,0.719020,0.754995
Cash ETF,2923.000000,0.249931,0.019385,0.002624,0.001640,-0.75,-1.07,9.30,-0.002085,595.000000,0.303243,0.977585,0.000219,0.000247,1.501662,3.812876,0.516593,0.000765,-0.000438,0.018791,0.022403,-0.00,-0.001724,0.00,0.175982,-0.719365,0.011309,-0.009523


## 7. Beta, leverage, tail, and concentration diagnostics

This panel answers the core investment question: did performance come from leverage, benchmark beta,
statistically estimated residual alpha, or a few exceptional trading days? Alpha is diagnostic—not a
causal claim—and its robust t-stat appears in the CAPM table.

In [12]:
display(results.diagnostics.to_frame("Value"))
display(results.regressions.style.format("{:.3f}"))

portfolio_metrics = results.metrics.loc["Portfolio"]
print(f"Portfolio CAGR: {portfolio_metrics['CAGR']:.2%}")
print(f"Without the best 10 days: {portfolio_metrics['CAGR Without Best 10 Days']:.2%}")
print(f"Historical CVaR: {portfolio_metrics[f'Historical CVaR {config.var_confidence:.0%}']:.2%}")

,Value
Gross Exposure,1.0000
Net Risky Exposure,1.0000
Financing Weight at Rebalance,0.0000
Leveraged,False
Portfolio Beta,0.5176
Jensen Alpha,-0.0036
CAPM R-squared,0.8539
Approx. Annual Beta Return,0.0646
Sharpe Ratio,0.6190
CAGR,0.0800


,Beta,Jensen Alpha,R-squared,Alpha t-stat,Beta t-stat
Portfolio,0.518,-0.004,0.854,-0.305,45.170
US Equity,1.014,-0.005,0.992,-1.167,279.638
International Equity,0.834,-0.030,0.721,-1.128,38.732
US Bonds,0.035,-0.006,0.014,-0.405,1.723
Gold,0.063,0.100,0.005,2.049,2.432
Real Estate,0.817,-0.046,0.504,-1.091,15.686
Cash ETF,-0.000,-0.002,0.000,-4.656,-0.804


Portfolio CAGR: 8.00%
Without the best 10 days: 4.95%
Historical CVaR: 1.45%


## 8. Return and risk attribution

Carino-linked contributions sum to compounded total return. Euler component risks sum to annualized
portfolio volatility. Return leadership and risk-budget consumption can therefore be compared without
mixing incompatible units.

In [13]:
total_return = (1 + results.evaluation_returns["Portfolio"]).prod() - 1
linked_sum = results.return_contributions.sum()
risk_sum = results.risk_contributions["Component Risk"].sum()
portfolio_vol = results.metrics.loc["Portfolio", "Annualized Volatility"]

print(f"Linked contributions / total return: {linked_sum:.4%} / {total_return:.4%}")
print(f"Component risk / portfolio volatility: {risk_sum:.4%} / {portfolio_vol:.4%}")
display(results.return_contributions.to_frame())
display(results.risk_contributions.style.format("{:.2%}"))
results.figures["attribution"].show()

Linked contributions / total return: 144.5646% / 144.5646%
Component risk / portfolio volatility: 9.9916% / 9.8556%


,Linked Return Contribution
US Equity,0.7404
International Equity,0.2407
Gold,0.2280
Real Estate,0.1158
US Bonds,0.0848
Cash ETF,0.0360
Financing,0.0000


,Weight,Marginal Risk,Component Risk,Percent of Total Risk
US Equity,30.00%,16.73%,5.02%,50.25%
International Equity,15.00%,15.60%,2.34%,23.42%
Real Estate,10.00%,16.49%,1.65%,16.50%
Gold,10.00%,5.19%,0.52%,5.20%
US Bonds,25.00%,1.85%,0.46%,4.63%
Cash ETF,10.00%,0.00%,0.00%,0.00%


## 9. Rolling and subperiod analysis

Rolling 252-trading-day windows reveal whether full-sample statistics are stable. Calendar-year tables
surface regime dependence and unusually strong or weak subperiods.

In [14]:
display(results.calendar_years.xs("Portfolio", level="Series").style.format("{:.2%}"))
results.figures["rolling_sharpe"].show()
results.figures["rolling_volatility"].show()
results.figures["rolling_beta"].show()

,Return,Volatility,Sharpe,Max Drawdown,Beta
Year,,,,,
2015,-1.08%,8.13%,-9.98%,-7.65%,48.61%
2016,7.22%,7.39%,93.76%,-4.26%,50.53%
2017,13.02%,3.77%,302.58%,-1.76%,42.32%
2018,-4.07%,7.91%,-73.49%,-9.48%,44.21%
2019,19.50%,5.75%,276.71%,-2.36%,41.16%
2020,12.53%,19.17%,69.13%,-21.20%,55.49%
2021,11.57%,7.44%,150.21%,-3.69%,52.91%
2022,-13.97%,13.81%,-117.52%,-19.84%,53.97%
2023,14.62%,8.35%,107.29%,-7.40%,55.50%


## 10. Interactive performance dashboard

The same Plotly figure objects are used in Colab and the exported report, preventing notebook/report
metric drift.

In [15]:
for chart_name in [
    "scorecard", "growth", "drawdown", "risk_return", "correlation", "monthly_returns"
]:
    results.figures[chart_name].show()

## 11. Monthly return table

Each monthly cell and YTD total is compounded from daily simple returns.

In [16]:
monthly = monthly_return_matrix(results.evaluation_returns["Portfolio"])
display(monthly.style.format("{:.1%}").background_gradient(cmap="RdYlGn", axis=None))

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD
Year,,,,,,,,,,,,,
2015,1.1%,1.3%,-0.5%,0.2%,0.2%,-1.8%,0.6%,-3.3%,-1.1%,4.1%,-0.9%,-0.9%,-1.1%
2016,-2.0%,0.9%,4.6%,0.9%,-0.0%,2.0%,2.6%,-0.7%,0.2%,-2.0%,-0.6%,1.3%,7.2%
2017,1.7%,2.1%,0.2%,1.0%,0.8%,0.4%,1.6%,0.8%,0.6%,0.8%,1.3%,1.0%,13.0%
2018,2.0%,-3.1%,-0.0%,-0.0%,1.0%,0.0%,1.2%,0.9%,-0.4%,-3.8%,1.5%,-3.3%,-4.1%
2019,5.5%,1.3%,1.3%,1.5%,-2.1%,4.3%,0.4%,0.9%,0.7%,1.6%,0.8%,1.9%,19.5%
2020,0.6%,-3.7%,-8.9%,7.4%,3.0%,2.0%,4.2%,2.6%,-2.0%,-1.4%,6.2%,3.3%,12.5%
2021,-0.6%,0.6%,1.5%,3.3%,1.5%,0.5%,1.3%,1.2%,-3.0%,3.3%,-1.3%,2.9%,11.6%
2022,-3.8%,-1.2%,1.0%,-5.3%,-0.4%,-5.0%,4.6%,-3.4%,-6.9%,2.8%,5.9%,-2.5%,-14.0%
2023,5.9%,-3.1%,2.5%,0.9%,-1.2%,3.0%,2.1%,-1.8%,-3.7%,-1.3%,6.7%,4.4%,14.6%


## 12. Automated tear sheet and research artifacts

This exports the custom interactive HTML report, all clean CSV tables, the exact run configuration,
and—when its plotting dependencies are compatible—an additional QuantStats report.

In [17]:
artifacts = export_analysis_artifacts(results, include_quantstats=True)
for name, path in artifacts.items():
    print(f"{name:24s} -> {path.resolve()}")

tear_sheet = artifacts["tear_sheet"]
display(HTML(f'<a href="{tear_sheet}" target="_blank"><b>Open interactive tear sheet</b></a>'))

if IN_COLAB:
    from google.colab import files
    files.download(str(tear_sheet))

adjusted_prices          -> /content/multi-asset-risk-terminal/outputs/adjusted_prices.csv
simple_returns           -> /content/multi-asset-risk-terminal/outputs/simple_returns.csv
log_returns              -> /content/multi-asset-risk-terminal/outputs/log_returns.csv
macro_features           -> /content/multi-asset-risk-terminal/outputs/macro_features.csv
performance_metrics      -> /content/multi-asset-risk-terminal/outputs/performance_metrics.csv
calendar_year_metrics    -> /content/multi-asset-risk-terminal/outputs/calendar_year_metrics.csv
capm_regressions         -> /content/multi-asset-risk-terminal/outputs/capm_regressions.csv
risk_contributions       -> /content/multi-asset-risk-terminal/outputs/risk_contributions.csv
monthly_portfolio_returns -> /content/multi-asset-risk-terminal/outputs/monthly_portfolio_returns.csv
return_contributions     -> /content/multi-asset-risk-terminal/outputs/linked_return_contributions.csv
diagnostics              -> /content/multi-asset-risk-termi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Interpretation checklist and next steps

Before presenting results, ask:

1. Is the Sharpe ratio persistent across rolling windows and calendar years?
2. Does CAPM beta and \(R^2\) explain most of the return?
3. Is Jensen alpha economically meaningful, and is its robust t-stat credible?
4. Does gross exposure exceed 100%, and is financing treated consistently?
5. Which assets dominate the volatility budget versus linked total return?
6. How much CAGR disappears when the ten best days are removed?
7. Are drawdown, CVaR, and downside capture acceptable for the mandate?
8. Are ETF proxies, survivorship, fees, liquidity, taxes, and data licensing suitable for the use case?

Professional upgrades include ALFRED vintages, multi-factor attribution, transaction costs, stress
testing, bootstrap confidence intervals, optimization constraints, institutional data adapters, and a
scheduled Streamlit/Dash deployment.